# 01c. Sampling 2025 to 2026 videos

**Goal:** build a manageable dataset of about 60,000 videos from the Hugging Face dataset
`datasocial/tiktok-5.6B-videos` (460 GB, 5.6 billion rows), without downloading it all.

**Method:** for each month from 2025-01 to 2026-06 (18 months), read 12 random row groups
from the monthly parquet file, in small batches to protect memory, and keep 3,333 random videos.
The result is 59,994 rows covering 546 distinct days.

**Why this period:** virality is relative and changes over time, so recent data is more
representative. We stop at 2026-06 so every video has had at least 4 months to collect views.

## Settings
- `PER_MONTH`: videos kept per month (same number per month, so no month dominates)
- `N_GROUPS`: row groups read per month (more groups means a better spread over the month)
- `BATCH`: rows read at a time (small, to avoid running out of RAM)

In [1]:
from pathlib import Path
import random
import pyarrow as pa
import pyarrow.parquet as pq
import polars as pl
from huggingface_hub import HfFileSystem

REPO = "datasocial/tiktok-5.6B-videos"
fs = HfFileSystem()

ROOT = Path.cwd().parent
TMP = ROOT / "data" / "sample_recent" / "monthly"
TMP.mkdir(parents=True, exist_ok=True)

PER_MONTH = 3_333
N_GROUPS = 12
BATCH = 20_000
SEED = 42

c:\Users\LENOVO\Documents\tiktok-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def sample_month(m):
    out_file = TMP / f"{m}.parquet"
    if out_file.exists():
        print(m, "already done")
        return
    random.seed(SEED)
    with fs.open(f"datasets/{REPO}/data/{m}.parquet", "rb") as fh:
        pf = pq.ParquetFile(fh)
        k = min(N_GROUPS, pf.num_row_groups)
        groups = random.sample(range(pf.num_row_groups), k)
        per_group = PER_MONTH // k + 50
        kept = []
        for g in groups:
            n_rows = pf.metadata.row_group(g).num_rows
            frac = min(1.0, per_group / n_rows)
            for batch in pf.iter_batches(batch_size=BATCH, row_groups=[g]):
                d = pl.from_arrow(pa.Table.from_batches([batch]))
                n = min(d.height, max(1, round(d.height * frac)))
                kept.append(d.sample(n=n, seed=SEED))
    d = pl.concat(kept, how="vertical_relaxed")
    d = d.sample(n=min(PER_MONTH, d.height), seed=SEED)
    d.write_parquet(out_file)
    print(m, "groups:", k, "of", pf.num_row_groups, "| kept:", d.height)

In [3]:
import time
t = time.time()
sample_month("2025-01")
print(round(time.time() - t), "seconds")

2025-01 groups: 12 of 74 | kept: 3333
177 seconds


In [4]:
MONTHS = [f"2025-{m:02d}" for m in range(1, 13)] + [f"2026-{m:02d}" for m in range(1, 7)]

for m in MONTHS:
    try:
        sample_month(m)
    except Exception as e:
        print(m, "FAILED:", e)

2025-01 already done
2025-02 groups: 12 of 68 | kept: 3333
2025-03 groups: 12 of 86 | kept: 3333
2025-04 groups: 12 of 89 | kept: 3333
2025-05 groups: 12 of 108 | kept: 3333
2025-06 groups: 12 of 133 | kept: 3333
2025-07 groups: 12 of 198 | kept: 3333
2025-08 groups: 12 of 152 | kept: 3333
2025-09 groups: 12 of 120 | kept: 3333
2025-10 groups: 12 of 129 | kept: 3333
2025-11 groups: 12 of 127 | kept: 3333
2025-12 groups: 12 of 130 | kept: 3333
2026-01 groups: 12 of 150 | kept: 3333
2026-02 groups: 12 of 137 | kept: 3333
2026-03 groups: 12 of 134 | kept: 3333
2026-04 groups: 12 of 139 | kept: 3333
2026-05 groups: 12 of 236 | kept: 3333
2026-06 groups: 12 of 175 | kept: 3333


## Combine the monthly files
Each month is saved separately, so a crash never loses finished months.
Here the 18 files are merged into `recent_60k_2025_26.parquet`.
Check: about 60k rows, and many distinct days.

**Finding:** `width` and `height` are almost never recorded (0.8%), so they will not be used as features.

In [6]:
files = sorted(TMP.glob("*.parquet"))
recent = pl.concat([pl.read_parquet(f) for f in files], how="vertical_relaxed").sort("posted_at")
recent.write_parquet(ROOT / "data" / "sample_recent" / "recent_60k_2025_26.parquet")
print(recent.shape)
print((recent["width"] > 0).mean(), (recent["country"] != "").mean())
print(recent["posted_at"].dt.truncate("1d").n_unique(), "distinct days")

(59994, 31)
0.0077674434110077675 1.0
546 distinct days
